In [14]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [15]:
X_train_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\train_featured.csv')
X_test_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\test_featured.csv')
y_train = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_train.csv')
y_test = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_test.csv')

### With Class Imbalance

In [16]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_feat, y_train)
y_pred = model.predict(X_test_feat)
y_prob = model.predict_proba(X_test_feat)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, model.predict_proba(X_test_feat)[:, 1]))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98     11390
           1       0.83      0.71      0.77      1108

    accuracy                           0.96     12498
   macro avg       0.90      0.85      0.87     12498
weighted avg       0.96      0.96      0.96     12498

ROC AUC Score: 0.9828291648573864


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


#### RandomSearch CV

In [17]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

log_reg = LogisticRegression(max_iter=5000)

param_dist = [
    
    # L2 regularization (most common)
    {
        'penalty': ['l2'],
        'C': uniform(0.001, 10),
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'class_weight': [None, 'balanced']
    },
    
    # L1 regularization
    {
        'penalty': ['l1'],
        'C': uniform(0.001, 10),
        'solver': ['liblinear', 'saga'],
        'class_weight': [None, 'balanced']
    },
    
    # Elastic Net
    {
        'penalty': ['elasticnet'],
        'C': uniform(0.001, 10),
        'solver': ['saga'],
        'l1_ratio': uniform(0, 1),
        'class_weight': [None, 'balanced']
    }
]

random_search = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=param_dist,
    n_iter=50,                 # number of random combinations
    scoring='roc_auc',         # important for credit risk
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)


random_search.fit(X_train_feat, y_train)

# Best model
best_model = random_search.best_estimator_

print("Best Parameters:", random_search.best_params_)
print("Best CV AUC:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'C': np.float64(2.2803516254194167), 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV AUC: 0.9830133284552642


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [18]:
from sklearn.metrics import roc_auc_score

y_pred_proba = best_model.predict_proba(X_test_feat)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print(classification_report(y_test, best_model.predict(X_test_feat)))
print("Test AUC:", test_auc)

              precision    recall  f1-score   support

           0       0.99      0.92      0.96     11390
           1       0.55      0.95      0.70      1108

    accuracy                           0.93     12498
   macro avg       0.77      0.94      0.83     12498
weighted avg       0.96      0.93      0.94     12498

Test AUC: 0.9829927924615613


### Without Class Imbalance

In [19]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_feat_sm, y_train_sm = smote.fit_resample(X_train_feat, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_sm.value_counts())

Before SMOTE: default
0          34298
1           3189
Name: count, dtype: int64
After SMOTE: default
0          34298
1          34298
Name: count, dtype: int64


In [20]:
from sklearn.linear_model import LogisticRegression
model_sm = LogisticRegression(max_iter=1000, random_state=42)
model_sm.fit(X_train_feat_sm, y_train_sm)
y_pred = model_sm.predict(X_test_feat)
y_prob = model_sm.predict_proba(X_test_feat)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.99      0.93      0.96     11390
           1       0.57      0.94      0.71      1108

    accuracy                           0.93     12498
   macro avg       0.78      0.93      0.84     12498
weighted avg       0.96      0.93      0.94     12498

ROC AUC Score: 0.9829343936507735


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


#### RandomSearch CV

In [21]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

log_reg = LogisticRegression(max_iter=5000)

param_dist = [
    
    # L2 regularization (most common)
    {
        'penalty': ['l2'],
        'C': uniform(0.001, 10),
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'class_weight': [None, 'balanced']
    },
    
    # L1 regularization
    {
        'penalty': ['l1'],
        'C': uniform(0.001, 10),
        'solver': ['liblinear', 'saga'],
        'class_weight': [None, 'balanced']
    },
    
    # Elastic Net
    {
        'penalty': ['elasticnet'],
        'C': uniform(0.001, 10),
        'solver': ['saga'],
        'l1_ratio': uniform(0, 1),
        'class_weight': [None, 'balanced']
    }
]

random_search = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=param_dist,
    n_iter=50,                 # number of random combinations
    scoring='roc_auc',         # important for credit risk
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)


random_search.fit(X_train_feat_sm, y_train_sm)

# Best model
best_model_sm = random_search.best_estimator_

print("Best Parameters:", random_search.best_params_)
print("Best CV AUC:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Best Parameters: {'C': np.float64(0.20684494295802447), 'class_weight': 'balanced', 'l1_ratio': np.float64(0.7219987722668247), 'penalty': 'elasticnet', 'solver': 'saga'}
Best CV AUC: 0.9854728776142346


In [22]:
from sklearn.metrics import roc_auc_score

best_model_sm.fit(X_train_feat_sm, y_train_sm)
y_pred = best_model_sm.predict(X_test_feat)
y_pred_proba = best_model_sm.predict_proba(X_test_feat)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print("Test AUC:", test_auc)
print(classification_report(y_test, y_pred))

d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Test AUC: 0.9829573728300522
              precision    recall  f1-score   support

           0       0.99      0.93      0.96     11390
           1       0.57      0.94      0.71      1108

    accuracy                           0.93     12498
   macro avg       0.78      0.93      0.83     12498
weighted avg       0.96      0.93      0.94     12498



In [23]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)
print("AUC:", auc)

# Try flipping probabilities
auc_flipped = roc_auc_score(y_test, 1 - y_prob)
print("AUC flipped:", auc_flipped)

AUC: 0.9829343936507735
AUC flipped: 0.017065606349226475


### Optuna

In [24]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer,f1_score
def objective(trial):
    
    params= {
        'C': trial.suggest_float('C', 1e-4, 1e4, log=True),
        'solver':trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'sag', 'liblinear', 'saga']),
        'tol': trial.suggest_float('tol', 1e-5, 1e-2, log=True),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
    }

    model = LogisticRegression(max_iter=10000, random_state=42, **params)
    f1_scorer = make_scorer(f1_score, average='macro')
    scores = cross_val_score(model, X_train_feat_sm, y_train_sm, cv=5, scoring=f1_scorer, n_jobs=-1)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-02-28 20:25:08,875] A new study created in memory with name: no-name-c5441d98-44ee-4333-94b1-30addbabfe77
[I 2026-02-28 20:25:09,385] Trial 0 finished with value: 0.9442682088358945 and parameters: {'C': 4.045716368436471, 'solver': 'sag', 'tol': 0.0005667442227997253, 'class_weight': None}. Best is trial 0 with value: 0.9442682088358945.
[I 2026-02-28 20:25:09,955] Trial 1 finished with value: 0.9442524114368229 and parameters: {'C': 0.2176645365535422, 'solver': 'sag', 'tol': 1.7610918894807056e-05, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9442682088358945.
[I 2026-02-28 20:25:10,638] Trial 2 finished with value: 0.9443265831890233 and parameters: {'C': 1.9970675642613618, 'solver': 'sag', 'tol': 1.0942172386843947e-05, 'class_weight': None}. Best is trial 2 with value: 0.9443265831890233.
[I 2026-02-28 20:25:10,838] Trial 3 finished with value: 0.9440367083901162 and parameters: {'C': 166.35303558614532, 'solver': 'lbfgs', 'tol': 0.0022823527236154797, 'cla

In [25]:
print("Best trial:")
trial = study.best_trial
print(' F1 Score: {}'.format(trial.value))
print('Parameters: ')
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

best_model_optuna = LogisticRegression(max_iter=10000, random_state=42, **trial.params)
best_model_optuna.fit(X_train_feat_sm, y_train_sm)
y_pred_optuna = best_model_optuna.predict(X_test_feat)
print(classification_report(y_test, y_pred_optuna))

d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Best trial:
 F1 Score: 0.9444135493785264
Parameters: 
    C: 0.5590896606426013
    solver: saga
    tol: 1.8024321997001805e-05
    class_weight: balanced
              precision    recall  f1-score   support

           0       0.99      0.93      0.96     11390
           1       0.57      0.94      0.71      1108

    accuracy                           0.93     12498
   macro avg       0.78      0.93      0.84     12498
weighted avg       0.96      0.93      0.94     12498



In [30]:
import joblib
import os
os.makedirs('D:\\Credit Risk Modelling\\models', exist_ok=True)
joblib.dump(model_sm, "D:\\Credit Risk Modelling\\models\\logistic_model.joblib")

['D:\\Credit Risk Modelling\\models\\logistic_model.joblib']